In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df['Delivery_Time'].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({'Delivery_Time'})")
plt.xlabel('Delivery_Time')
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

In [ ]:
#since we have 106 null values with the target and we have a big data we are gonna remove the null values
df = df.dropna(subset=['Delivery_Time'])
#now we check again
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()

    df[col] = le.fit_transform(df[col])

df.head()

#this code below is for onehotencoder but i didnt like having 750 columns so i would just use labelencoding
#onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
#X_encoded = pd.DataFrame(onehot_encoder.fit_transform(df), columns=onehot_encoder.get_feature_names_out(df.columns))
#X_encoded.head()


In [ ]:
# Task 5: Write your code here:
#we drop the target column because we dont want to scale it
numerical_cols =  df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# TODO: Apply fit_transform to scale the numerical columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 6: Write your code here:
#we already plotted the target distrubtion so we can see there if it is imbalanced or not

In [ ]:
# Task 1: Write your code here:
#dropping the target column for X so X now only contains features
X = df.drop('Delivery_Time', axis=1)
#here we put the target column in y
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
models = {
  "Random Forest": RandomForestClassifier(
      n_estimators=200,
      max_depth=10,
      random_state=42
  )}


# Determine the number of classes for one-hot encoding
num_classes = len(np.unique(y))

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
model_losses = {}

print("Starting K-Fold Cross-Validation for each model...")

# 4. For each model in case we add more models:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_losses = [] # List to store loss from each fold

    for fold, (train_index, test_index) in enumerate(kf.split(X_train)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        # Train the current model
        model.fit(X_train, y_train)

        # Generate predictions (probabilities) on the validation set
        y_pred = model.predict_proba(X_test)

        # Convert y_val_fold (true labels) into a one-hot encoded format
        onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        y_val_one_hot = onehot_encoder(y_test, num_classes)

        # Calculate categorical cross-entropy loss for the current fold
        loss = mean_absolute_error(y_val_one_hot, y_pred)
        fold_losses.append(loss)

    # Calculate the average loss for the model
    avg_loss = np.mean(fold_losses)
    model_losses[model_name] = avg_loss

    # Print the average cross-validation loss for the current model
    print(f"{model_name} - Average MAE loss: {avg_loss:.4f}")

print("\nAll models trained and evaluated. Stored average losses:")
print(model_losses)



In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Exam Scores (Ground Truth)")
plt.ylabel("Predicted Exam Scores")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: